# Fast-volatility maximum: public proxy calibration

## tl;dr

The fixed `fast_volatility_max_v1` candidate is rejected without retuning. On 8,633 previously uninspected BTC five-minute conditions and 25,899 registered forecasts, it improved Brier score by `0.000177` and log loss by `0.001040`. Both chronological halves, all three decision offsets, the overconfidence tail, and both paired day-bootstrap intervals were directionally positive. However, the Brier gain missed the preregistered `0.0005` minimum, so no strategy variant, exact replay, runtime change, profitability claim, or A+ credit is authorized.

## Context & Methods

The current retained strategy is profitable but overconfident. This diagnostic asks whether one minimal probability-model change is strong enough to justify a later Polymarket opportunity screen: replace the current `max(0.30, one-hour realized volatility)` input with `max(0.30, one-hour realized volatility, existing causal fast EWMA volatility)`.

### Key assumptions

- The rule, two disjoint 15-day windows, decision offsets (`120`, `150`, `179` seconds), proper scores, deterministic day bootstrap, and every pass gate were frozen before these public-window labels were downloaded or inspected.
- The calculation reproduces the Rust engine's annualization, 15-minute-half-life EWMA recurrence, Black–Scholes `d2`, probability clamp, and Abramowitz–Stegun normal CDF approximation.
- Binance terminal direction is only a research proxy for Chainlink settlement. The population is every complete Binance five-minute window, not gated Polymarket opportunities; it cannot establish tradable edge or profit.
- Exact terminal ties are excluded. Each retained condition contributes exactly one forecast at each registered offset.
- A post-score integrity amendment corrects an overly broad rationale sentence: higher volatility can move an exactly at-the-money Black–Scholes probability slightly farther below `0.5`. The formula and every gate remain unchanged.

## Data

### 1. Re-run the registered calculation from checksum-verified archives

In [1]:
from pathlib import Path
import hashlib
import json
import os
import sys

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "rust_engine").exists():
    ROOT = Path("../..").resolve()
assert (ROOT / "rust_engine").exists(), ROOT

ARCHIVE_DIR = Path(os.environ.get(
    "POLYMOMENTUM_FAST_VOL_ARCHIVE_DIR",
    "/private/tmp/polymomentum-fast-volatility-max-20260721/binance_1s",
))
REGISTRY = ROOT / "deploy/promotions/evidence/strategy_registry"
PREREGISTRATION = REGISTRY / "20260721_fast_volatility_max_preregistration.json"
EVIDENCE = REGISTRY / "20260721_fast_volatility_max_public_calibration.json"
SNAPSHOT = REGISTRY / "source_snapshots/20260721_fast_volatility_max_forecasts.jsonl.gz"
AMENDMENT = REGISTRY / "20260721_fast_volatility_max_integrity_amendment.json"

sys.path.insert(0, str(ROOT / "scripts"))
import analyze_fast_volatility_max as analysis

assert ARCHIVE_DIR.exists(), (
    "Download the 31 checksum-pinned official archives listed in the saved evidence "
    "or set POLYMOMENTUM_FAST_VOL_ARCHIVE_DIR."
)
evidence = analysis.run(ARCHIVE_DIR, EVIDENCE, SNAPSHOT)
preregistration = json.loads(PREREGISTRATION.read_text())
amendment = json.loads(AMENDMENT.read_text())
print({
    "registration_status": preregistration["status"],
    "result_status": evidence["status"],
    "archives": evidence["source_data_quality"]["archives"],
    "rows": evidence["source_data_quality"]["rows"],
})

{'registration_status': 'PREREGISTERED_BEFORE_PUBLIC_WINDOW_DOWNLOAD_OR_LABEL_INSPECTION', 'result_status': 'PUBLIC_PROXY_CALIBRATION_REJECTED_NO_RETUNING', 'archives': 31, 'rows': 2678400}


### 2. Verify source and forecast grain

In [2]:
source_quality = evidence["source_data_quality"]
forecast_quality = evidence["forecast_data_quality"]
quality_rows = [
    {"layer": "source", "check": "Adjacent SHA-256 failures", "observed": source_quality["checksum_failures"]},
    {"layer": "source", "check": "Timestamp duplicates / regressions / gaps", "observed": f'{source_quality["timestamp_duplicates"]} / {source_quality["timestamp_regressions"]} / {source_quality["one_second_gap_violations"]}'},
    {"layer": "source", "check": "Invalid prices / close durations", "observed": f'{source_quality["invalid_prices"]} / {source_quality["invalid_close_durations"]}'},
    {"layer": "forecast", "check": "Retained conditions", "observed": forecast_quality["retained_conditions"]},
    {"layer": "forecast", "check": "Retained forecasts", "observed": forecast_quality["retained_registered_forecasts"]},
    {"layer": "forecast", "check": "Complete registered forecast fraction", "observed": forecast_quality["complete_registered_forecasts_fraction"]},
    {"layer": "forecast", "check": "Duplicate condition-offset rows", "observed": forecast_quality["duplicate_condition_offsets"]},
    {"layer": "forecast", "check": "Invalid probabilities", "observed": forecast_quality["invalid_probabilities"]},
]
pd.DataFrame(quality_rows)

,layer,check,observed
0,source,Adjacent SHA-256 failures,0
1,source,Timestamp duplicates / regressions / gaps,0 / 0 / 0
2,source,Invalid prices / close durations,0 / 0
3,forecast,Retained conditions,8633
4,forecast,Retained forecasts,25899
5,forecast,Complete registered forecast fraction,0.99919
6,forecast,Duplicate condition-offset rows,0
7,forecast,Invalid probabilities,0


## Results

### 3. Compare proper scores across the frozen populations

In [3]:
overall = evidence["results"]["overall"]
comparison_rows = [{
    "population": "overall",
    "conditions": overall["conditions"],
    "forecasts": overall["forecasts"],
    "brier_improvement": overall["brier_improvement"],
    "log_loss_improvement": overall["log_loss_improvement"],
    "candidate_active_fraction": overall["candidate_volatility_active_fraction"],
}]
for name, score in evidence["results"]["chronological_windows"].items():
    comparison_rows.append({
        "population": name,
        "conditions": score["conditions"],
        "forecasts": score["forecasts"],
        "brier_improvement": score["brier_improvement"],
        "log_loss_improvement": score["log_loss_improvement"],
        "candidate_active_fraction": score["candidate_volatility_active_fraction"],
    })
for name, score in evidence["results"]["decision_offsets"].items():
    comparison_rows.append({
        "population": f"offset_{name}s",
        "conditions": score["conditions"],
        "forecasts": score["forecasts"],
        "brier_improvement": score["brier_improvement"],
        "log_loss_improvement": score["log_loss_improvement"],
        "candidate_active_fraction": score["candidate_volatility_active_fraction"],
    })
tail = evidence["results"]["overconfidence_tail"]
comparison_rows.append({
    "population": "baseline_probability_le_0.25_or_ge_0.75",
    "conditions": tail["conditions"],
    "forecasts": tail["forecasts"],
    "brier_improvement": tail["brier_improvement"],
    "log_loss_improvement": tail["log_loss_improvement"],
    "candidate_active_fraction": tail["candidate_volatility_active_fraction"],
})
comparison = pd.DataFrame(comparison_rows)
comparison

,population,conditions,forecasts,brier_improvement,log_loss_improvement,candidate_active_fraction
0,overall,8633,25899,0.000177,0.001040,0.186416
1,fresh_holdout,4316,12948,0.000127,0.000879,0.200880
2,older,4317,12951,0.000227,0.001202,0.171956
3,offset_120s,8633,8633,0.000245,0.001183,0.184872
4,offset_150s,8633,8633,0.000164,0.000976,0.187305
5,offset_179s,8633,8633,0.000122,0.000962,0.187073
6,baseline_probability_le_0.25_or_ge_0.75,5338,11673,0.000356,0.002224,0.249036


### 4. Apply the preregistered gates without reinterpretation

In [4]:
bootstrap = evidence["results"]["bootstrap"]
gate_rows = [
    {"gate": name, "passed": passed}
    for name, passed in evidence["gate_evaluation"]["checks"].items()
]
display(pd.DataFrame(gate_rows))
print({
    "brier_day_bootstrap_95pct": bootstrap["brier_improvement_95pct"],
    "log_loss_day_bootstrap_95pct": bootstrap["log_loss_improvement_95pct"],
    "failed_checks": evidence["gate_evaluation"]["failed_checks"],
    "decision": evidence["decision"],
})

,gate,passed
0,source_checksum_failures_zero,True
1,source_timestamp_duplicates_zero,True
2,source_timestamp_regressions_zero,True
3,source_gaps_at_most_two_seconds,True
4,source_invalid_prices_zero,True
5,minimum_conditions_each_half,True
6,complete_registered_forecasts_at_least_99pct,True
7,overall_brier_improvement_at_least_0_0005,False
8,overall_log_loss_improvement_at_least_0_001,True
9,brier_bootstrap_lower_bound_positive,True


{'brier_day_bootstrap_95pct': [9.361693906398474e-05, 0.0002684271129953332], 'log_loss_day_bootstrap_95pct': [0.000645582189020129, 0.001472579068910676], 'failed_checks': ['overall_brier_improvement_at_least_0_0005'], 'decision': {'public_proxy_calibration_passed': False, 'strategy_variant_authorized': False, 'exact_strategy_replay_authorized': False, 'runtime_change_authorized': False, 'paper_or_live_trading_authorized': False, 'profitability_claim': False, 'a_plus_claim': False, 'next_step': 'Reject fast_volatility_max_v1 and do not tune this family on the registered windows.'}}


### 5. Validate reproducibility and the post-score wording correction

In [5]:
def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

for source in preregistration["source_pins_before_evaluation"].values():
    assert sha256_file(ROOT / source["path"]) == source["sha256"]
assert sha256_file(PREREGISTRATION) == evidence["authority"]["preregistration_sha256"]
assert evidence["status"] == "PUBLIC_PROXY_CALIBRATION_REJECTED_NO_RETUNING"
assert evidence["gate_evaluation"]["failed_checks"] == ["overall_brier_improvement_at_least_0_0005"]
assert evidence["decision"]["strategy_variant_authorized"] is False
assert evidence["decision"]["runtime_change_authorized"] is False
assert evidence["decision"]["profitability_claim"] is False
assert evidence["decision"]["a_plus_claim"] is False
assert amendment["status"] == "POST_SCORE_WORDING_CORRECTION_FORMULA_AND_GATES_UNCHANGED"
assert all(not changed for changed in amendment["frozen_contract"].values())
assert amendment["correction"]["observed_scope"]["candidate_more_confident_forecasts"] == 15
print({
    "evidence_sha256": sha256_file(EVIDENCE),
    "snapshot_sha256": sha256_file(SNAPSHOT),
    "candidate_more_confident_forecasts": 15,
    "maximum_absolute_confidence_increase": amendment["correction"]["observed_scope"]["maximum_absolute_confidence_increase"],
    "validation": "PASS",
})

{'evidence_sha256': '4673836c3e0086dda271eb9f12817c216a63cf0e3e4ec0a28297ed7bf9c5474d', 'snapshot_sha256': '618aa9a8cc18c9647fd447bcfed1cea161b4f2d8d876c185f25b2fc3f102953d', 'candidate_more_confident_forecasts': 15, 'maximum_absolute_confidence_increase': 4.6722093762774364e-05, 'validation': 'PASS'}


## Takeaways

- **Reject `fast_volatility_max_v1` without tuning.** Its `0.000177` Brier improvement is positive but only 35.4% of the fixed `0.0005` minimum. The log-loss gate passes at `0.001040`, and every stability/uncertainty check passes, but the contract requires all gates.
- **The effect is broad but too small.** Candidate volatility activates on `18.64%` of forecasts, improves both chronological halves and every registered offset, and improves the overconfidence-tail Brier/log-loss by `0.000356 / 0.002224`. This is credible diagnostic information, not enough effect size for another expensive strategy branch.
- **Do not convert this into a strategy adjustment.** Binance direction is not the official settlement label and these are not Polymarket opportunities. No exact replay, runtime field, paper/live change, profitability claim, or A+ credit follows.
- **Preserve the active decision order.** Finish the sealed binary-complement block at 750 conditions and score it once. The official-anchor contract remains the only later model-specification candidate, already carrying negative but underpowered retrospective evidence.